# COVID-19 Global Trend Analysis

**Author:** Muhammad Masoom Khan  
**Dataset:** Johns Hopkins CSSE + Our World in Data  
**Period:** January 2020 – December 2022

---

I put this together to get a clearer picture of how COVID spread globally and whether vaccination actually made a dent in mortality. There's a lot of noise in this data, especially in the early months when testing was inconsistent, so I spent a decent amount of time cleaning before doing any real analysis.

## 1. Load Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# I prefer this style for dark backgrounds — cleaner for presentations
plt.style.use('dark_background')

PURPLE = '#6c5ce7'
BLUE   = '#6495ed'
GREEN  = '#00b894'
RED    = '#e17055'
ORANGE = '#fdcb6e'

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("Libraries loaded.")

## 2. Load & Reshape Data

JHU stores the data in wide format — each date is its own column. I need to melt it into long format (one row per country per day) before I can do anything useful with it.

In [ ]:
confirmed_raw = pd.read_csv('data/time_series_covid19_confirmed_global.csv')
deaths_raw    = pd.read_csv('data/time_series_covid19_deaths_global.csv')

print(f"Confirmed: {confirmed_raw.shape[0]} rows, {confirmed_raw.shape[1]} columns")
print(f"Deaths:    {deaths_raw.shape[0]} rows, {deaths_raw.shape[1]} columns")
confirmed_raw.head(3)

In [ ]:
def reshape_jhu(df, value_name):
    # The first 4 cols are metadata, rest are dates
    id_cols   = ['Province/State', 'Country/Region', 'Lat', 'Long']
    date_cols = [c for c in df.columns if c not in id_cols]
    
    long = df.melt(id_vars=id_cols, value_vars=date_cols,
                   var_name='date', value_name=value_name)
    long['date'] = pd.to_datetime(long['date'])
    
    # Some countries have multiple rows (provinces) — sum them
    grouped = (long.groupby(['Country/Region', 'date'])[value_name]
                   .sum().reset_index()
                   .rename(columns={'Country/Region': 'country'}))
    return grouped

confirmed = reshape_jhu(confirmed_raw, 'confirmed')
deaths    = reshape_jhu(deaths_raw,    'deaths')

df = confirmed.merge(deaths, on=['country', 'date'], how='left')
df = df.sort_values(['country', 'date'])

# Daily new cases from cumulative — clip negatives (data corrections in source)
df['new_cases']  = df.groupby('country')['confirmed'].diff().fillna(0).clip(lower=0)
df['new_deaths'] = df.groupby('country')['deaths'].diff().fillna(0).clip(lower=0)
df['cfr']        = (df['deaths'] / df['confirmed'].replace(0, np.nan) * 100).round(2)

print(f"Merged dataset: {df.shape[0]:,} rows")
print(f"Countries: {df['country'].nunique()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## 3. Global Trends

First pass — just want to see the overall shape of the pandemic. Rolling averages are essential here; the raw daily counts are too noisy to read.

In [ ]:
global_d = df.groupby('date')[['new_cases', 'new_deaths', 'confirmed', 'deaths']].sum().reset_index()
global_d['cases_7d']  = global_d['new_cases'].rolling(7).mean()
global_d['deaths_7d'] = global_d['new_deaths'].rolling(7).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True, facecolor='#1a1d23')
ax1.set_facecolor('#1a1d23')
ax2.set_facecolor('#1a1d23')

ax1.bar(global_d['date'], global_d['new_cases'], color=BLUE, alpha=0.2, width=1)
ax1.plot(global_d['date'], global_d['cases_7d'], color=BLUE, lw=2.5, label='7-day rolling avg')
ax1.set_ylabel('New Cases', fontsize=12)
ax1.set_title('Global COVID-19 Daily New Cases  (2020–2022)', fontsize=14, fontweight='bold', pad=12)
ax1.legend(facecolor='#2f3436', labelcolor='white')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# Mark the main variant waves — these dates are approximate
waves = [('2021-01-15','Alpha'), ('2021-07-01','Delta'), ('2022-01-01','Omicron')]
for ds, lbl in waves:
    ax1.axvline(pd.to_datetime(ds), color='white', ls='--', alpha=0.35, lw=1)
    ax1.text(pd.to_datetime(ds), ax1.get_ylim()[1]*0.82, lbl,
             fontsize=9, color='white', ha='center', alpha=0.7)

ax2.bar(global_d['date'], global_d['new_deaths'], color=RED, alpha=0.2, width=1)
ax2.plot(global_d['date'], global_d['deaths_7d'], color=RED, lw=2.5, label='7-day rolling avg')
ax2.set_ylabel('New Deaths', fontsize=12)
ax2.set_title('Global COVID-19 Daily Deaths  (2020–2022)', fontsize=14, fontweight='bold', pad=12)
ax2.legend(facecolor='#2f3436', labelcolor='white')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=35)

plt.tight_layout(h_pad=0.5)
plt.savefig('outputs/01_global_trends.png', dpi=150, bbox_inches='tight', facecolor='#1a1d23')
plt.show()
print("Saved: outputs/01_global_trends.png")

## 4. Country Comparison

Per-capita numbers would be more fair for comparison, but total counts tell the story of where the absolute burden fell. Pakistan is highlighted in purple throughout.

In [ ]:
latest = df.groupby('country').last().reset_index()
top12 = latest.nlargest(12, 'confirmed')

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor='#1a1d23')
for ax in axes:
    ax.set_facecolor('#1a1d23')

# Total cases
bar_colors = [PURPLE if c == 'Pakistan' else BLUE for c in top12['country']]
bars = axes[0].barh(top12['country'], top12['confirmed']/1e6,
                    color=bar_colors, edgecolor='none', height=0.65)
axes[0].set_xlabel('Total Confirmed Cases (Millions)', fontsize=11)
axes[0].set_title('Top 12 Countries — Total Cases', fontsize=13, fontweight='bold')
axes[0].invert_yaxis()
for bar, val in zip(bars, top12['confirmed']/1e6):
    axes[0].text(val + 0.4, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}M', va='center', fontsize=9, color='white')

# CFR
top12_cfr = latest.dropna(subset=['cfr']).nlargest(12, 'deaths')
bar_colors2 = [PURPLE if c == 'Pakistan' else RED for c in top12_cfr['country']]
axes[1].barh(top12_cfr['country'], top12_cfr['cfr'],
             color=bar_colors2, edgecolor='none', height=0.65)
mean_cfr = top12_cfr['cfr'].mean()
axes[1].axvline(mean_cfr, color='white', ls='--', alpha=0.5, lw=1.5,
                label=f'Avg CFR: {mean_cfr:.1f}%')
axes[1].set_xlabel('Case Fatality Rate (%)', fontsize=11)
axes[1].set_title('Case Fatality Rate — Top 12 by Deaths', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()
axes[1].legend(facecolor='#2f3436', labelcolor='white')

plt.tight_layout()
plt.savefig('outputs/02_country_comparison.png', dpi=150, bbox_inches='tight', facecolor='#1a1d23')
plt.show()

## 5. Vaccination Impact

This is the question I was most interested in — does vaccination actually show up in the CFR numbers? The scatter plot below suggests yes, fairly strongly.

In [ ]:
try:
    vacc = pd.read_csv('data/vaccinations_owid.csv')
    vacc['date'] = pd.to_datetime(vacc['date'])
    vacc_l = (vacc.groupby('location').last().reset_index()
              [['location','people_fully_vaccinated_per_hundred']].dropna()
              .rename(columns={'location':'country',
                               'people_fully_vaccinated_per_hundred':'vacc_pct'}))

    combined = latest.merge(vacc_l, on='country').dropna(subset=['cfr','vacc_pct'])
    combined = combined[combined['confirmed'] > 10000]

    fig, ax = plt.subplots(figsize=(12, 8), facecolor='#1a1d23')
    ax.set_facecolor('#1a1d23')

    sc = ax.scatter(combined['vacc_pct'], combined['cfr'],
                    c=combined['confirmed']/1e6, cmap='plasma',
                    s=100, alpha=0.75, edgecolors='white', linewidth=0.4)
    cb = plt.colorbar(sc, ax=ax)
    cb.set_label('Total Cases (Millions)', color='white')
    cb.ax.yaxis.set_tick_params(color='white')
    plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

    z   = np.polyfit(combined['vacc_pct'], combined['cfr'], 1)
    xr  = np.linspace(0, 85, 200)
    ax.plot(xr, np.poly1d(z)(xr), '--', color=GREEN, lw=2, alpha=0.8, label='Trend line')

    for _, row in combined[combined['country'].isin(
            ['Pakistan','United States','United Kingdom','India','Brazil','Germany'])].iterrows():
        ax.annotate(row['country'], (row['vacc_pct'], row['cfr']),
                    textcoords='offset points', xytext=(8, 4), fontsize=9, color='#ddd')

    r = combined[['vacc_pct','cfr']].corr().iloc[0,1]
    ax.set_xlabel('Population Fully Vaccinated (%)', fontsize=12)
    ax.set_ylabel('Case Fatality Rate (%)', fontsize=12)
    ax.set_title(f'Vaccination Rate vs. Case Fatality Rate\nPearson r = {r:.2f}  — negative correlation confirms vaccination impact',
                 fontsize=13, fontweight='bold')
    ax.legend(facecolor='#2f3436', labelcolor='white')

    plt.tight_layout()
    plt.savefig('outputs/03_vaccination_impact.png', dpi=150, bbox_inches='tight', facecolor='#1a1d23')
    plt.show()
    print(f"Correlation: {r:.3f}")

except FileNotFoundError:
    print("Vaccination file not found — check data/README_data.md for download link")

## 6. Summary Stats

In [ ]:
total_cases  = global_d['new_cases'].sum()
total_deaths = global_d['new_deaths'].sum()
peak_row     = global_d.loc[global_d['cases_7d'].idxmax()]
global_cfr   = total_deaths / total_cases * 100

print("=" * 55)
print("  COVID-19 GLOBAL ANALYSIS — KEY FINDINGS")
print("=" * 55)
print(f"  Total cases in dataset  :  {total_cases:>15,.0f}")
print(f"  Total deaths in dataset :  {total_deaths:>15,.0f}")
print(f"  Overall case fatality   :  {global_cfr:>14.2f}%")
print(f"  Peak 7-day avg (date)   :  {peak_row['date'].strftime('%b %Y')}")
print(f"  Peak 7-day avg (cases)  :  {peak_row['cases_7d']:>15,.0f} / day")
print()
print("  Top 5 countries by total cases:")
print(latest.nlargest(5,'confirmed')[['country','confirmed','deaths','cfr']]
      .to_string(index=False))
print()
print("  Charts saved to outputs/")